# 3. Coordinate Transformation

In [ ]:
# Main imports
import sys
from pathlib import Path

# Add the project root to the Python path
PROJECT_ROOT = Path().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT))

# Import custom modules
from src.config import PROJECT_ROOT, DATA_ROOT, initialize_h5
from src.utilities import (
    read_denoise_results,
    save_coord_transform_results,
    get_keys_h5,
    get_attrs_h5,
    repack_h5,
)
from src.processing import transform_spring_pendulum_coordinates, generate_spring_pendulum_equations, generate_spring_pendulum_candidates

In [ ]:
# Directory and files setup

TYPE = 'tangle_01'
STAGE = '03_coord_transform'

INPUT_PATH = DATA_ROOT / TYPE / '02_denoise.h5'
OUTPUT_PATH = DATA_ROOT / TYPE / (STAGE + '.h5')

print('INPUT_PATH: \t{:}'.format(INPUT_PATH))
print('OUTPUT_PATH: \t{:}'.format(OUTPUT_PATH))
print()

# Initialize output file
overwrite = True
verbose = True

initialize_h5(OUTPUT_PATH, overwrite=overwrite, verbose=verbose)

In [ ]:
# Utility functions

def print_verbose(params, coord_data, equations, equations_data, candidates_data=None):
    print('='*80)
    print(f"{params['exp_name'].ljust(10)} ({params['comments']})")
    print('-'*80)
    print('Coordinate transformation: cartesian -> spherical (relative to piston)')
    
    # Print coordinate data info
    print('Coordinate ranges:')
    if 'x' in coord_data: print(f'  x:  [{coord_data["x"].min():.3f}, {coord_data["x"].max():.3f}]')
    if 'y' in coord_data: print(f'  y:  [{coord_data["y"].min():.3f}, {coord_data["y"].max():.3f}]')
    if 'z' in coord_data: print(f'  z:  [{coord_data["z"].min():.3f}, {coord_data["z"].max():.3f}]')
    if 'r' in coord_data: print(f'  r (radius) : [{coord_data["r"].min():.3f}, {coord_data["r"].max():.3f}]')
    if 'theta' in coord_data: print(f'  θ (polar)  :  [{coord_data["theta"].min():.3f}, {coord_data["theta"].max():.3f}] (rad)')
    if 'phi' in coord_data: print(f'  φ (azimuth): [{coord_data["phi"].min():.3f}, {coord_data["phi"].max():.3f}] (rad)')

    # Print feature generation info
    print('Equations: {:}'.format(equations))
    for eq_name, eq_data in equations_data.items():
        print(f'  {eq_name}:')
        print(f'    features: {eq_data["feature_names"]}  {eq_data["feature_data"].shape}')
        print(f'    target: {eq_data["target_name"]} {eq_data["target_data"].shape}')
    
    # Print candidate data info
    if candidates_data is not None:
        print('Candidates:')
        print(f'  feature: {candidates_data["feature_candidate_names"]}')
        print(f'  target:  {candidates_data["target_candidate_names"]}')
    print()

## Compute Coordinate Transformations

In [ ]:
# Coordinate transformation and equation data configuration

equations = ['z_equation']
coord_transform_config = {
    'transform': transform_spring_pendulum_coordinates.__name__,
    'derivative_method': 'finite_difference_derivative',
    'equations': equations,
    'include_candidates': False,
    'coord_names': ['x', 'y', 'z', 'r', 'theta', 'phi'],
    'coord_labels': {
        'x': r'$x$',
        'y': r'$y$',
        'z': r'$z$',
        'r': r'$r$',
        'theta': r'$\theta$',
        'phi': r'$\phi$'
    }
}

In [ ]:
# Compute coordinate transformations
data_labels = get_keys_h5(INPUT_PATH)

for data_label in data_labels:
        
    # 1. Read trajectories and params (denoised)
    datasets, params, denoise_config, coordinate_cv_results = read_denoise_results(INPUT_PATH, data_label)

    # 2. Transform coordinates to spring-pendulum coordinate system
    coord_data = transform_spring_pendulum_coordinates(datasets)

    # 3. Generate features and targets for spring-pendulum regression
    equations_data = generate_spring_pendulum_equations(coord_data, equations=equations)

    # 4. Generate candidates for spring-pendulum regression
    if coord_transform_config['include_candidates']:
        candidates_data = generate_spring_pendulum_candidates(coord_data)
    else:
        candidates_data = None

    if verbose: print_verbose(params, coord_data, equations, equations_data, candidates_data)

    # 4. Save transformed coordinates and features 
    coord_datasets = {'t': coord_data['t']} | {coord_key: coord_data[coord_key] for coord_key in coord_transform_config['coord_names']}
    save_coord_transform_results(
        OUTPUT_PATH, data_label, 
        coord_datasets, params, 
        coord_transform_config, equations_data,
        candidates_data=candidates_data,
        hdf5_options={'compression':'gzip', 'compression_opts':9}
    )

repack_h5(OUTPUT_PATH, in_place=True)

## Time Interval and Coordinate Range Tuning

In [ ]:
# Time interval configuration for each experiment

time_interval_config = {
    'tw0_low': {
        'time_range': (18, 100),
        'interval_sep': 3,
        'interval_len': 3,
        'sample_interval': (30, 35)
    },
    'tw2_opt': {
        'time_range': (0,100),
        'interval_sep': 3,
        'interval_len': 3,
        'sample_interval': (18, 23)
    },
    'tw10_opt': {
        'time_range': (0, 15),
        'interval_sep': 2,
        'interval_len': 2,
        'sample_interval': (6, 11)
    },
}

# Add default parameters for any other experiments
data_labels = get_keys_h5(INPUT_PATH)
for data_label in data_labels:
    if data_label not in time_interval_config:
        time_interval_config[data_label] = {
            'time_range': (0, 100),
            'interval_sep': 3,
            'interval_len': 5,
            'sample_interval': (5, 10)
        }

In [ ]:
# Coordinate limit configuration

coord_limit_config = {
    # low
    'tw10_low': {
        'feature_limits': {
            'r': [0.00, 0.06],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.1]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
    'tw2_low': {
        'feature_limits': {
            'r': [0.00, 0.06],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
    'tw0_low': {
        'feature_limits': {
            'r': [0.00, 0.06],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },

    # opt
    'tw10_opt': {
        'feature_limits': {
            'r': [0.00, 0.1],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 80]
        }
    },
    'tw2_opt': {
        'feature_limits': {
            'r': [0.02, 0.1],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
    'tw0_opt': {
        'feature_limits': {
            'r': [0.02, 0.1],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },

    # high
    'tw10_high': {
        'feature_limits': {
            'r': [0.01, 0.07],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
    'tw2_high': {
        'feature_limits': {
            'r': [0.03, 0.08],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
    'tw0_high': {
        'feature_limits': {
            'r': [0.0, 0.1],
            'r_dot': [-2, 2],
            'cos_theta': [-1.01, -0.4]
        },
        'target_limits': {
            'z_ddot': [-30, 50]
        }
    },
}

# Add default limits for any other experiments
for data_label in data_labels:
    if data_label not in coord_limit_config:
        coord_limit_config[data_label] = {
            'feature_limits': {
                'r': [0, 0.1],
                'r_dot': [-2, 2],
                'cos_theta': [-1.1, 0.1]
            },
            'target_limits': {
                'z_ddot': [-50, 80]
            }
        }

In [ ]:
# Save the time interval and coordinate limit configurations
import json
import h5py
from src.utilities import to_serializable

with h5py.File(OUTPUT_PATH, 'a') as f:
    f.attrs['time_interval_config'] = json.dumps(to_serializable(time_interval_config))
    f.attrs['coord_limit_config'] = json.dumps(to_serializable(coord_limit_config))

print(f"Saved configs as attrs in {OUTPUT_PATH}.")
print(f"  time_interval_config: {list(time_interval_config.keys())}")
print(f"  coord_limit_config: {list(coord_limit_config.keys())}")
print()

# Usage
attrs = get_attrs_h5(OUTPUT_PATH)
time_interval_config = json.loads(attrs['time_interval_config'])
coord_limit_config = json.loads(attrs['coord_limit_config'])